In [ ]:
from google.colab import files

uploaded = files.upload()


Saving london_property_ml.csv to london_property_ml.csv


In [ ]:
import pandas as pd


df = pd.read_csv("london_property_ml.csv")
df

,sale_date,property_type,tenure,price_gbp,postcode,transactionid,lmk_key,floor_area_m2,number_rooms,borough
0,2024-02-05,F,L,400000,SE15 3BL,{1C3E72FE-AF7C-F410-E063-4804A8C0DFA6},aaaf3c2a84a1d89fb9bc3c40135fc203d7b0313c5e59d2...,82.00,3.0,Southwark
1,2021-03-26,T,F,1435000,SW6 5AR,{C6209F5E-CD7D-295E-E053-6C04A8C0DDCC},c3fdd972d033d32bb228719f82a0bc80c4c5459f6561f8...,139.00,6.0,Hammersmith and Fulham
2,2002-12-18,T,F,312500,N16 6DG,{EE8B4D8A-9901-4034-92C5-A3FBB375CE6D},1699752249962019022116483522188491,144.00,7.0,Hackney
3,2003-07-07,S,F,149995,DA7 4JN,{F9F5B44E-DD73-409A-AAE8-26D8227E981D},858751139402012112117495200322598,100.00,5.0,Bexley
4,2012-07-13,F,L,500000,N6 5YW,{9B206793-FB86-4027-8CCC-85A758E53742},250539466352009032420180106210258,65.40,3.0,Islington
...,...,...,...,...,...,...,...,...,...,...
99995,1998-06-19,T,F,62500,RM8 2UJ,{D0116479-9E4D-4F96-AE42-5958A201C82E},285660600922009051411383631048691,76.30,4.0,Barking and Dagenham
99996,2003-05-30,F,L,100000,DA8 3SP,{C760E82B-5B69-459E-9AD8-02DBA8A2B202},1517743929062017020911571049538653,59.00,3.0,Bexley
99997,2018-07-31,T,F,329000,N9 7LJ,{75050A85-6CCC-9A88-E053-6B04A8C02390},1622171429062018041008150977218888,70.00,5.0,Enfield
99998,2010-06-23,T,F,230000,HA4 6PX,{A6001EE0-CF2A-4617-88F9-89C2FED906D0},482762879702010051015592574509908,61.44,4.0,Hillingdon


In [ ]:
!pip install pgeocode -q

In [ ]:
import pgeocode
import pandas as pd

In [ ]:
nomi = pgeocode.Nominatim("GB")

In [ ]:
unique_postcodes = (
    df["postcode"]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .unique()
)

len(unique_postcodes)

57889

In [ ]:
postcode_geo = nomi.query_postal_code(unique_postcodes)

postcode_geo = postcode_geo[
    ["postal_code", "latitude", "longitude"]
].copy()


postcode_geo = postcode_geo.rename(
    columns={
        "postal_code": "postcode"
    }
)





In [ ]:
df = df[
    (df["price_gbp"] >= 50_000) &
    (df["price_gbp"] <= 5_000_000) &
    (df["floor_area_m2"] >= 20) &
    (df["floor_area_m2"] <= 500) &
    (df["number_rooms"] >= 1) &
    (df["number_rooms"] <= 15)
].copy()

print(df.shape)

(89568, 10)


In [ ]:
import pandas as pd

# Postcode district
df["postcode_district"] = (
    df["postcode"]
    .str.upper()
    .str.split()
    .str[0]
)

# Date
df["sale_date"] = pd.to_datetime(df["sale_date"])

df["sale_year"] = df["sale_date"].dt.year
df["sale_month"] = df["sale_date"].dt.month

In [ ]:
df[
    [
        "price_gbp",
        "floor_area_m2",
        "number_rooms",
        "postcode_district",
        "borough",
        "property_type",
        "tenure",
        "sale_year"
    ]
].head(10)

,price_gbp,floor_area_m2,number_rooms,postcode_district,borough,property_type,tenure,sale_year
0,400000,82.00,3.0,SE15,Southwark,F,L,2024
1,1435000,139.00,6.0,SW6,Hammersmith and Fulham,T,F,2021
2,312500,144.00,7.0,N16,Hackney,T,F,2002
3,149995,100.00,5.0,DA7,Bexley,S,F,2003
4,500000,65.40,3.0,N6,Islington,F,L,2012
5,515000,97.64,6.0,KT4,Sutton,S,F,2015
6,365000,102.76,6.0,NW10,Hammersmith and Fulham,T,F,2010
8,305000,83.00,4.0,CR0,Croydon,F,L,2022
9,895000,98.00,3.0,SW13,Richmond upon Thames,F,L,2014
10,82950,55.10,3.0,DA14,Bexley,F,L,1999


In [ ]:
features = [
    "floor_area_m2",
    "number_rooms",
    "postcode_district",
    "borough",
    "property_type",
    "tenure",
    "sale_year"
]

X = df[features]

y = df["price_gbp"]

In [ ]:
X.head()
y.head()

,price_gbp
0,400000
1,1435000
2,312500
3,149995
4,500000


In [ ]:
numerical_features = [
    "floor_area_m2",
    "number_rooms",
    "sale_year"
]

categorical_features = [
    "postcode_district",
    "borough",
    "property_type",
    "tenure"
]

In [ ]:
df["postcode"] = (
    df["postcode"]
    .astype(str)
    .str.upper()
    .str.strip()
)

postcode_geo["postcode"] = (
    postcode_geo["postcode"]
    .astype(str)
    .str.upper()
    .str.strip()
)



df = df.merge(
    postcode_geo,
    on="postcode",
    how="left"
)




In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

(71654, 7)
(17914, 7)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['floor_area_m2',
                                                   'number_rooms',
                                                   'sale_year']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['postcode_district',
                                                   'borough', 'property_type',
                                                   'tenure'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred = model.predict(X_test)


results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

results.head(40)

,Actual,Predicted
80230,200000,417418.004480
66989,225000,325487.754315
82866,106500,39384.078476
87006,315000,441754.576431
9798,570000,588902.199568
96430,391667,599284.957182
7986,350000,689612.410155
27287,499999,452494.923068
10979,152500,554937.883739
57451,308000,493264.054748


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred)

rmse = mean_squared_error(
    y_test,
    y_pred
) ** 0.5

r2 = r2_score(y_test, y_pred)

print(f"MAE: £{mae:,.0f}")
print(f"RMSE: £{rmse:,.0f}")
print(f"R²: {r2:.3f}")

MAE: £128,752
RMSE: £231,011
R²: 0.646


In [ ]:
import joblib

joblib.dump(
    model,
    "london_property_model.pkl"
)

['london_property_model.pkl']

In [ ]:
from google.colab import files

files.download("london_property_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import joblib

model = joblib.load("london_property_model.pkl")

print(model)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['floor_area_m2',
                                                   'number_rooms',
                                                   'sale_year']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['postcode_district',
                                                   'borough', 'property_type',
                                                   'tenure'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=20, min_samples_leaf=3,
                                       n_estimators=300, n_jobs=-1,
                                       random_state=42))])


In [ ]:
price = model.predict(house)[0]

print(f"Estimated value: £{price:,.0f}")

NotFittedError: Pipeline is not fitted yet.